# Joint SED and spectrum fit of SDSS J211920.33+004436.6

This notebook collects broadband photometry with [BandWagon](https://github.com/burke86/bandwagon), downloads the SDSS spectrum, and fits both with one `jaxsedfit` model. The photometry and the spectrum share the host and AGN continuum; the spectrum adds tied emission lines, Fe II and the Balmer continuum from the detailed spectral model.

You need internet access, `bandwagon`, and the DSPS SSP file `tempdata.h5` in the repository root.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpyro.distributions as dist
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.sdss import SDSS
from bandwagon import matches_to_photometry, xmatch_catalogs

from jaxsedfit import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GalaxyConfig,
    InferenceConfig,
    JAXSEDFit,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    SpectroscopyData,
    SpectrumConfig,
    load_filter_curves,
    plot_corner,
)

## 1. Photometry

BandWagon cross-matches GALEX (AUTO/Kron fluxes), SDSS (PSF magnitudes) and AllWISE. `photometry_method` records how each flux was measured, which the host-capture model uses: total-flux methods capture all the host light, PSF photometry only part of it.

In [ ]:
coord = SkyCoord("21h19m20.33s +00d44m36.6s")
object_id = "SDSS_J211920.33+004436.6"

matches = xmatch_catalogs(coord, source_id=[object_id], cache=False)
phot = matches_to_photometry(matches, max_mag_err=1.0)
phot = phot[np.isfinite(phot["flux_mjy"]) & (phot["flux_err_mjy"] > 0)]
phot["catalog", "filter_name", "photometry_method", "flux_mjy", "flux_err_mjy", "psf_fwhm_arcsec"]

## 2. SDSS spectrum

`jaxsedfit` works in $f_\nu$ (mJy), so the SDSS $f_\lambda$ spectrum is converted with `astropy.units`. We fit 3600–9000 Å, where the SDSS spectrophotometry is reliable.

In [ ]:
matches_spec = SDSS.query_region(coord, spectro=True, radius=5 * u.arcsec)
hdul = SDSS.get_spectra(matches=matches_spec[:1])[0]
spec = hdul[1].data
z = hdul[2].data["Z"][0]
resolving_power = 1 / (2.355 * np.log(10) * 1e-4 * np.median(spec["wdisp"]))

wave = 10 ** spec["loglam"]
keep = (spec["ivar"] > 0) & (wave > 3600) & (wave < 9000)
flambda = u.Unit("1e-17 erg s-1 cm-2 AA-1")
to_mjy = u.spectral_density(wave[keep] * u.AA)
flux_mjy = (spec["flux"][keep] * flambda).to_value(u.mJy, to_mjy)
err_mjy = (1 / np.sqrt(spec["ivar"][keep]) * flambda).to_value(u.mJy, to_mjy)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(wave[keep], flux_mjy, lw=0.6)
ax.set(xlabel="Observed wavelength [Å]", ylabel="Flux density [mJy]", title=f"{object_id}, z = {z:.3f}");

## 3. Configure the joint fit

`SpectroscopyData` adds the spectrum with its instrument, fiber aperture and resolving power; `SpectrumConfig` chooses the spectral components. `fit_spectrum_scale=True` fits a gray scale between the spectrum and the photometry, with a prior width set by `spectrum_scale_prior_sigma_dex`. `fit_host_kinematics=True` fits the stellar velocity dispersion, and `host_sfh_model="delayed_burst"` adds a recent burst to the delayed-$\tau$ star-formation history.

In [ ]:
filter_names = list(phot["filter_name"])
config = FitConfig(
    observation=Observation(
        object_id=object_id, redshift=z, ra=coord.ra.deg, dec=coord.dec.deg, apply_mw_deredden=True
    ),
    photometry=PhotometryData(
        filter_names=filter_names,
        fluxes=list(phot["flux_mjy"]),
        errors=list(phot["flux_err_mjy"]),
        psf_fwhm_arcsec=list(phot["psf_fwhm_arcsec"]),
        photometry_method=list(phot["photometry_method"]),
    ),
    filters=FilterSet(curves=load_filter_curves(filter_names)),
    spectroscopy=SpectroscopyData(
        wave_obs=wave[keep],
        fluxes=flux_mjy,
        errors=err_mjy,
        instrument="SDSS",
        aperture_diameter_arcsec=3.0,
        resolving_power=resolving_power,
    ),
    galaxy=GalaxyConfig(
        dsps_ssp_fn="../tempdata.h5", n_wave=512, fit_host_kinematics=True, host_sfh_model="delayed_burst"
    ),
    agn=AGNConfig(agn_type=1, line_flux_scale_mjy=0.1),
    spectrum=SpectrumConfig(
        power_law_enabled=True,
        host_enabled=True,
        lines_enabled=True,
        tied_lines=True,
        feii_enabled=True,
        balmer_continuum_enabled=True,
    ),
    likelihood=LikelihoodConfig(
        use_host_capture_model=True,
        systematics_width=0.08,
        fit_spectrum_scale=True,
        spectrum_scale_prior_sigma_dex=0.1,
    ),
    inference=InferenceConfig(
        method="optax+nuts", map_steps=4000, learning_rate=5e-3, seed=3, num_warmup=200, num_samples=200
    ),
)

# Line-width and burst priors
config.prior_config.agn.log_broad_line_width_kms = dist.TruncatedNormal(
    np.log(3000.0), 0.4, low=np.log(1000.0), high=np.log(15000.0)
)
config.prior_config.agn.log_narrow_line_width_kms = dist.TruncatedNormal(
    np.log(500.0), 0.3, low=np.log(100.0), high=np.log(1500.0)
)
config.prior_config.host.log_sfh_burst_age_gyr = dist.Uniform(np.log(0.01), np.log(0.5))
config.prior_config.host.log_sfh_burst_tau_gyr = dist.Uniform(np.log(0.01), np.log(0.2))

## 4. Fit

200 warmup and 200 posterior draws demonstrate the workflow; use longer, multi-chain runs with convergence checks for science, and `method="optax"` while iterating on the setup.

In [ ]:
fitter = JAXSEDFit(config)
result = fitter.fit(progress_bar=False)

## 5. Results

`result.predict().spectrum` gives typed posterior results for the spectrum: the model per observation and the named line components and line groups.

In [ ]:
spectrum = result.predict(n_draws=200).spectrum
print("Posterior model shape (draws, pixels):", spectrum.observations[0].model_flux_mjy.shape)
print("Line groups:", sorted(spectrum.line_groups))

In [ ]:
fitter.plot_spectrum(show_plot=True);

In [ ]:
fitter.plot_sed(show=True);

Positive parameters sampled in log space are converted back before plotting.

In [ ]:
samples = result.samples
plot_corner(
    {
        r"$\alpha_{\rm AGN}$": samples["pl_slope"],
        r"$E(B-V)_{\rm host}$": np.exp(samples["log_ebv_gal"]),
        r"$E(B-V)_{\rm AGN}$": np.exp(samples["log_ebv_agn"]),
        r"$\log M_\star$": samples["log_stellar_mass"],
        r"$f_{\rm cov}$": samples["fcov"],
        r"$f_{\rm hot}$": np.exp(samples["log_hot_fcov"]),
    },
    max_params=None,
    show=True,
);